In [6]:
# --- 1. Imports ---
import time
import re
import json
import os
import pandas as pd
import numpy as np
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

In [2]:
# --- 2. KeywordAxisInfer Implementation (Fallback/Simplified) ---
class KeywordAxisInfer:
    """
    KeywordAxisInfer 클래스 (임시 구현)
    
    원래 모델 코드(Neural Network)가 없으므로, 
    'student_distilled_e5_rule/config_runtime.json'에 정의된 
    키워드 규칙(RULES)을 사용하여 점수를 계산하는 방식으로 구현했습니다.
    
    만약 딥러닝 모델(student_head.pt)을 사용해야 한다면, 
    이 클래스 내용을 원래 사용하시던 코드로 교체해주세요.
    """
    def __init__(self, model_path):
        self.model_path = model_path
        self.config = self._load_config()
        self.axes = self.config.get("AXES", [])
        self.rules = self.config.get("RULES", {})
        
        print(f"Loaded KeywordAxisInfer with {len(self.axes)} axes from {model_path}")

    def _load_config(self):
        config_path = os.path.join(self.model_path, "config_runtime.json")
        if not os.path.exists(config_path):
            print(f"Warning: Config file not found at {config_path}")
            return {}
        with open(config_path, "r", encoding="utf-8") as f:
            return json.load(f)

    def infer(self, texts):
        scores = []
        labels = []
        
        for text in texts:
            text_str = str(text)
            row_scores = []
            
            # 각 축(axis)별로 키워드가 포함되어 있는지 확인
            for axis in self.axes:
                keywords = self.rules.get(axis, [])
                # 단순 키워드 매칭: 하나라도 있으면 1, 없으면 0
                matched = any(keyword in text_str for keyword in keywords)
                score = 1.0 if matched else 0.0
                row_scores.append(score)
            
            scores.append(row_scores)
            labels.append(row_scores) # 라벨도 동일하게 0/1로 반환
            
        return np.array(scores), np.array(labels)

In [3]:
# --- 3. Crawler Functions ---
def detect_platform(url):
    if "a-bly.com" in url:
        return "ably"
    elif "musinsa.com" in url:
        return "musinsa"
    elif "zigzag.kr" in url:
        return "zigzag"
    else:
        raise ValueError("지원하지 않는 플랫폼입니다.")

class MusinsaPerfectScraper:
    def __init__(self):
        chrome_options = Options()
        chrome_options.add_argument("--headless")
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")
        chrome_options.add_argument("--disable-gpu")
        chrome_options.add_argument("user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
        self.driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

    def run(self, url):
        self.result = {}
        try:
            self.driver.get(url)
            wait = WebDriverWait(self.driver, 15)
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div[class*='FixedArea__Inner']")))
            self.driver.execute_script("window.scrollTo(0, 800);")
            time.sleep(2)

            soup = BeautifulSoup(self.driver.page_source, 'html.parser')
            fixed = soup.select_one("div[class*='FixedArea__Inner']")

            if fixed:
                self.result['product_name'] = self._text(fixed, 'span[class*="GoodsName"]')
                self.result['rating'] = self._text(fixed, 'div[class*="ReviewSummary__Wrap"] span[class*="text-body"]')
                self.result['review_count'] = self._text(fixed, 'div[class*="ReviewSummary__Wrap"] span[class*="underline"]')
                self.result['discount_rate'] = self._text(fixed, 'span[class*="Price__DiscountRate"]')
                
                arrival_info = fixed.select_one('div[class*="PlusDeliveryArrivalInfo__Wrapper"]')
                self.result['shipping_guarantee'] = arrival_info.get_text(" ", strip=True) if arrival_info else ""
                self.result['product_likes'] = self._text(fixed, 'div[class*="Like__Container"] span')

            return self.result
        except Exception as e:
            print(f"❌ Musinsa Error: {e}")
            return {}

    def _text(self, parent, selector):
        tag = parent.select_one(selector)
        return tag.get_text(strip=True) if tag else ""

    def close(self):
        self.driver.quit()

class ZigzagDetailCrawler:
    def __init__(self):
        self.chrome_options = Options()
        self.chrome_options.add_argument('--headless')
        self.chrome_options.add_argument('--no-sandbox')
        self.chrome_options.add_argument('--disable-dev-shm-usage')
        self.chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")
        self.driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=self.chrome_options)

    def _expand_product_info(self):
        try:
            more_button = WebDriverWait(self.driver, 3).until(
                EC.presence_of_element_located((By.XPATH, "//button[contains(., '상품정보 더 보기')]"))
            )
            self.driver.execute_script("arguments[0].click();", more_button)
            time.sleep(2)
        except Exception:
            pass

    def _safe_get_text(self, soup, selector):
        element = soup.select_one(selector)
        return element.get_text(strip=True) if element else None

    def crawl_detail(self, url):
        self.driver.get(url)
        time.sleep(3)
        self._expand_product_info()
        soup = BeautifulSoup(self.driver.page_source, 'html.parser')
        data = {}
        data['name'] = self._safe_get_text(soup, 'div.pdp__title h1')
        data['discount_rate'] = self._safe_get_text(soup, 'div[class*="css-1fwo2a0"]')
        
        shipping_list = []
        is_zdelivery = soup.select_one('svg[data-zds-graphic="LogoZdelivery"]')
        if is_zdelivery:
            shipping_list.append("직진배송")
        data['shipping_info'] = " | ".join(shipping_list) if shipping_list else None
        data['review_rating'] = self._safe_get_text(soup, 'span[class*="eic0mh2"]')
        data['review_count'] = self._safe_get_text(soup, 'span[class*="zds4_lh8eqt5"]')
        return data

    def close(self):
        self.driver.quit()

def crawl_ably(url):
    options = Options()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')
    options.add_argument('--window-size=1920,2000')
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36")
    prefs = {"profile.managed_default_content_settings.images": 2}
    options.add_experimental_option("prefs", prefs)
    options.add_argument('--log-level=3')

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    try:
        driver.get(url)
        time.sleep(5)
        for scroll in [1000, 2000]:
            driver.execute_script(f"window.scrollTo(0, {scroll});")
            time.sleep(1.5)

        product_data = {}
        try:
            title_tag = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, 'h2, p.typography__body1')))
            product_data['product_name'] = title_tag.text
        except:
            product_data['product_name'] = "제목 없음"

        try:
            price_container = driver.find_element(By.CLASS_NAME, "sc-ad5f1e6f-0")
            discount_rate_el = price_container.find_elements(By.CLASS_NAME, "color__pink30")
            product_data['discount_rate'] = discount_rate_el[0].text.replace('%', '').strip() if discount_rate_el else "0"
        except:
             product_data['discount_rate'] = "0"

        try:
            page_text = driver.find_element(By.TAG_NAME, "body").text
            count_match = re.search(r'리뷰\s*([\d,]+)개', page_text)
            if count_match:
                product_data['review_count'] = int(count_match.group(1).replace(',', ''))
            else:
                product_data['review_count'] = 0
            
            score_match = re.search(r'([\d.]+)%가\s*만족한', page_text)
            if score_match:
                raw_percent = float(score_match.group(1))
                product_data['review_score'] = round(max(0, (raw_percent - 50) / 10), 2)
            else:
                product_data['review_score'] = 0.0
        except:
            product_data['review_count'] = 0
            product_data['review_score'] = 0.0

        try:
            likes_container = driver.find_element(By.CLASS_NAME, "sc-45b21edb-3")
            likes_text = likes_container.find_element(By.CLASS_NAME, "color__pink30").text.strip()
            if '만' in likes_text:
                product_data['product_likes'] = str(int(float(likes_text.replace('만', '')) * 10000))
            else:
                 product_data['product_likes'] = likes_text.replace(',', '')
        except:
            product_data['product_likes'] = "0"

        product_data['is_direct_shipping'] = 1 if '오늘출발' in page_text else 0
        return product_data
    except Exception as e:
        print(f"Ably Error: {e}")
        return {}
    finally:
        driver.quit()

def crawl_product_data(url, platform):
    print(f"Crawling {url} on {platform}...")
    data = {}
    
    if platform == "musinsa":
        crawler = MusinsaPerfectScraper()
        try:
            data = crawler.run(url)
        finally:
            crawler.close()
    elif platform == "zigzag":
        crawler = ZigzagDetailCrawler()
        try:
            data = crawler.crawl_detail(url)
        finally:
            crawler.close()
    elif platform == "ably":
        data = crawl_ably(url)
    else:
        return {}

    # Normalize Data
    normalized_data = {}
    normalized_data["product_name"] = data.get("product_name") or data.get("name") or "Unknown"
    
    dr = data.get("discount_rate", "0")
    if isinstance(dr, str): dr = re.sub(r'[^0-9]', '', dr)
    normalized_data["discount_rate"] = int(dr) if dr else 0
    
    rs = data.get("review_score") or data.get("rating") or data.get("review_rating") or "0"
    if isinstance(rs, str): rs = re.sub(r'[^0-9.]', '', rs)
    normalized_data["review_score"] = float(rs) if rs else 0.0
    
    rc = data.get("review_count", "0")
    if isinstance(rc, str): rc = re.sub(r'[^0-9]', '', rc)
    normalized_data["review_count"] = int(rc) if rc else 0
    
    likes = data.get("product_likes") or data.get("brand_likes") or "0"
    if isinstance(likes, str):
        if '만' in likes: likes = str(int(float(likes.replace('만', '')) * 10000))
        likes = re.sub(r'[^0-9]', '', likes)
    normalized_data["product_likes"] = int(likes) if likes else 0
    
    shipping = data.get("is_direct_shipping")
    if shipping is None:
        shipping_info = str(data.get("shipping_info") or "")
        if "직진" in shipping_info or "오늘출발" in shipping_info: shipping = 1
        else: shipping = 0
    normalized_data["is_direct_shipping"] = int(shipping)
    
    return normalized_data

def extract_numeric_features(data):
    return {
        "discount_rate": int(data.get("discount_rate", 0)),
        "review_score": float(data.get("review_score", 0.0)),
        "review_count": int(data.get("review_count", 0)),
        "product_likes": int(data.get("product_likes", 0)),
    }

def extract_categorical_features(data, platform):
    return {
        "platform": platform,
        "is_direct_shipping": int(data.get("is_direct_shipping", 0))
    }


In [4]:
# --- 4. Main Feature Extraction (Integration) ---
def extract_features_from_url(url, sim_model=None):
    platform = detect_platform(url)
    product_data = crawl_product_data(url, platform)

    numeric_features = extract_numeric_features(product_data)
    categorical_features = extract_categorical_features(product_data, platform)
    
    # Binary Features using sim_model
    product_name = product_data.get("product_name", "")
    binary_features = {}
    
    if sim_model:
        try:
            scores, labels = sim_model.infer([product_name])
            # labels[0] is expected to be [1, 0, 1, ...] matching SIM_COLS
            SIM_COLS = [
                "sim_quality_logic",
                "sim_trend_hype",
                "sim_temptation",
                "sim_fit_anxiety",
                "sim_bundle",
                "sim_confidence"
            ]
            # Map index to column name
            for i, col in enumerate(SIM_COLS):
                if i < len(labels[0]):
                    binary_features[col] = int(labels[0][i])
                else:
                    binary_features[col] = 0
        except Exception as e:
             print(f"Model Inference Error: {e}")
             binary_features = {col: 0 for col in SIM_COLS}
    else:
        # Default if no model
        SIM_COLS = ["sim_quality_logic", "sim_trend_hype", "sim_temptation", "sim_fit_anxiety", "sim_bundle", "sim_confidence"]
        binary_features = {col: 0 for col in SIM_COLS}

    feature_dict = {
        **numeric_features,
        **categorical_features,
        **binary_features
    }

    return pd.DataFrame([feature_dict])

In [8]:
# --- 5. Execution ---
# 모델 로드 (상대 경로 사용)
model_path = "./student_distilled_e5_rule"
if os.path.exists(model_path):
    infer_model = KeywordAxisInfer(model_path)
else:
    print(f"Warning: Model path {model_path} not found.")
    infer_model = None

# 테스트
test_urls = [
     "https://m.a-bly.com/goods/3129076"
    # "https://zigzag.kr/catalog/products/122935080"
    # "https://www.musinsa.com/products/4457092"
]

results = []
for test_url in test_urls:
    print(f"\nProcessing: {test_url}")
    df_result = extract_features_from_url(test_url, sim_model=infer_model)
    results.append(df_result)

if results:
    final_df = pd.concat(results, ignore_index=True)
    print(final_df)

Loaded KeywordAxisInfer with 6 axes from ./student_distilled_e5_rule

Processing: https://m.a-bly.com/goods/3129076
Crawling https://m.a-bly.com/goods/3129076 on ably...
   discount_rate  review_score  review_count  product_likes platform   
0              0           3.9          1938          97000     ably  \

   is_direct_shipping  sim_quality_logic  sim_trend_hype  sim_temptation   
0                   1                  1               0               0  \

   sim_fit_anxiety  sim_bundle  sim_confidence  
0                0           0               0  
